# Construction Cost per m² — Competition-grade Starter Notebook

This notebook provides a reproducible, competition-focused pipeline for predicting construction cost per square meter using tabular economic/geographic features and satellite data (Sentinel-2, VIIRS). It includes: an overview, data loading, satellite feature extraction helpers, GroupKFold validation by location, a country-debias technique, and a LightGBM baseline.

Sections:
- Problem overview
- Install & imports
- Load tabular data (train + evaluation)
- Quick EDA & cleaning
- Satellite feature extraction helpers (Sentinel & VIIRS)
- Merge features & modeling (GroupKFold + country-debias)
- Train final model & export predictions

Run cells in order. Adapt filepaths for local image tiles if needed.

**IMPORTANT:** Use `GroupKFold` by location to avoid leakage. Also apply a country-level de-biasing (subtract country mean) so the model learns within-country variation rather than memorizing country-level costs.

In [30]:
# Install runtime dependencies (uncomment to run)
!pip install -q rasterio rioxarray lightgbm scikit-learn pandas matplotlib seaborn tqdm

# If running in restricted environment, install only what's necessary.


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
# Imports
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb
from tqdm.auto import tqdm

pd.options.display.max_columns = 200

In [32]:
# Paths
BASE = Path('d:/Projects/Construction-cost')
DATA_DIR = BASE / 'datasets'
TRAIN_TAB = DATA_DIR / 'train_tabular.csv'
EVAL_TAB = DATA_DIR / 'evaluation_tabular_no_target.csv'
SENTINEL_DIR = DATA_DIR / 'train_composite'  # adapt if needed
VIIRS_DIR = DATA_DIR / 'train_composite'     # placeholder; update if separate

print('BASE:', BASE)
print('TRAIN_TAB exists:', TRAIN_TAB.exists())
print('EVAL_TAB exists:', EVAL_TAB.exists())

BASE: d:\Projects\Construction-cost
TRAIN_TAB exists: True
EVAL_TAB exists: True


In [33]:
# Load tabular data
train = pd.read_csv(TRAIN_TAB)
eval_df = pd.read_csv(EVAL_TAB)

print('train shape', train.shape)
train.head()

train shape (1024, 23)


,data_id,geolocation_name,quarter_label,country,year,deflated_gdp_usd,us_cpi,developed_country,landlocked,region_economic_classification,access_to_airport,access_to_port,access_to_highway,access_to_railway,straight_distance_to_capital_km,seismic_hazard_zone,flood_risk_class,tropical_cyclone_wind_risk,tornadoes_wind_risk,koppen_climate_zone,sentinel2_tiff_file_name,viirs_tiff_file_name,construction_cost_per_m2_usd
0,LP81L,Dinagat Islands,2019-Q3,Philippines,2019,2.996821e+11,117.689915,No,No,Lower-middle income,No,Yes,Yes,No,770.0,Moderate,Yes,High,Very Low,Af,sentinel_2_dinagat_islands_2019-Q3.tif,viirs_dinagat_islands_2019-Q3.tif,129.997420
1,E7EOB,29000 Nara,2024-Q2,Japan,2024,3.928801e+12,143.968241,Yes,Yes,High income,No,No,Yes,Yes,370.0,Moderate,Yes,Low,Very Low,Cfa,sentinel_2_29000_nara_2024-Q2.tif,viirs_29000_nara_2024-Q2.tif,1567.878774
2,WAOUA,05000 Akita,2020-Q1,Japan,2020,4.069008e+12,118.435291,Yes,No,High income,Yes,Yes,Yes,Yes,450.0,Moderate,Yes,Low,Very Low,Dfa,sentinel_2_05000_akita_2020-Q1.tif,viirs_05000_akita_2020-Q1.tif,2009.827701
3,2IZ5P,Cotabato,2020-Q4,Philippines,2020,2.912443e+11,119.402476,No,Yes,Lower-middle income,Yes,No,Yes,No,870.0,High,Yes,Moderate,Very Low,Af,sentinel_2_cotabato_2020-Q4.tif,viirs_cotabato_2020-Q4.tif,377.279961
4,RJ5XF,Pampanga,2019-Q3,Philippines,2019,2.996821e+11,117.689915,No,Yes,Lower-middle income,Yes,No,Yes,Yes,65.0,Moderate,Yes,Moderate,Very Low,Am,sentinel_2_pampanga_2019-Q3.tif,viirs_pampanga_2019-Q3.tif,163.905688


In [34]:
# Quick EDA
display(train.describe(include='all').T)
print('Missing values (train):')
display(train.isna().sum().sort_values(ascending=False).head(30))

# Country counts and target by country
display(train.groupby('country')['construction_cost_per_m2_usd'].agg(['count','mean','std']).sort_values('mean', ascending=False))

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
data_id,1024,1024,0VGNX,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
geolocation_name,1024,125,18000 Fukui,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
quarter_label,1024,24,2021-Q3,72,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,1024,2,Japan,567,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,1024.0,NaN,NaN,NaN,2020.796875,1.571916,2019.0,2020.0,2021.0,2022.0,2024.0
deflated_gdp_usd,1024.0,NaN,NaN,NaN,2370260062297.703613,1854773352940.898926,291244296180.307739,299682074191.289368,3880440088794.578125,4069008481605.97998,4247588558872.259277
us_cpi,1024.0,NaN,NaN,NaN,125.200014,9.335148,115.978119,117.795699,120.683801,130.298326,144.727986
developed_country,1024,2,Yes,567,NaN,NaN,NaN,NaN,NaN,NaN,NaN
landlocked,1024,2,No,794,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region_economic_classification,1024,4,High income,567,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Missing values (train):


tropical_cyclone_wind_risk         4
geolocation_name                   0
data_id                            0
country                            0
year                               0
deflated_gdp_usd                   0
quarter_label                      0
us_cpi                             0
developed_country                  0
region_economic_classification     0
landlocked                         0
access_to_port                     0
access_to_highway                  0
access_to_railway                  0
access_to_airport                  0
straight_distance_to_capital_km    0
seismic_hazard_zone                0
flood_risk_class                   0
tornadoes_wind_risk                0
koppen_climate_zone                0
sentinel2_tiff_file_name           0
viirs_tiff_file_name               0
construction_cost_per_m2_usd       0
dtype: int64

,count,mean,std
country,,,
Japan,567,1830.268632,309.108275
Philippines,457,214.819797,71.979599


### Fix a few known issues and encode categoricals
- Fill the four missing `tropical_cyclone_wind_risk` with zeros or the country median (example below).
- Encode booleans/nominals as integers where appropriate.

In [35]:
# Example fixes
# tropical_cyclone_wind_risk is categorical (e.g., 'High','Low','Moderate'). Impute with mode.
if 'tropical_cyclone_wind_risk' in train.columns:
    mode_vals = train['tropical_cyclone_wind_risk'].mode()
    mode_val = mode_vals[0] if len(mode_vals)>0 else np.nan
    train['tropical_cyclone_wind_risk'] = train['tropical_cyclone_wind_risk'].fillna(mode_val)
    if 'tropical_cyclone_wind_risk' in eval_df.columns:
        eval_df['tropical_cyclone_wind_risk'] = eval_df['tropical_cyclone_wind_risk'].fillna(mode_val)
# Simple categorical encoding
for c in ['country','climate_zone']:
    if c in train.columns:
        train[c] = train[c].astype('category')
        eval_df[c] = eval_df[c].astype('category')
        train[c+'_cat'] = train[c].cat.codes
        eval_df[c+'_cat'] = eval_df[c].cat.codes

# Create an infra access score as an example
access_cols = [col for col in train.columns if col.startswith('access_')]
if access_cols:
    train['infra_access_score'] = train[access_cols].sum(axis=1)
    eval_df['infra_access_score'] = eval_df[access_cols].sum(axis=1)

train.shape

(1024, 25)

## Satellite feature extraction helpers
The functions below compute simple summary statistics from raster tiles: means, stds, percentiles, NDVI and VIIRS stats. They rely on `rasterio`. If your images are stored differently, adapt the file path pattern.

In [36]:
# Satellite extraction helpers (Sentinel-2: 12 bands assumed)
import rasterio
from rasterio.enums import Resampling

def raster_stats(path, bands=None, rescale=False):
    # returns dict of mean/std/p10/p90 per band
    stats = {}
    try:
        with rasterio.open(path) as src:
            nodata = src.nodata
            for i in (bands or src.indexes):
                arr = src.read(i).astype('float32')
                if nodata is not None:
                    arr = np.where(arr==nodata, np.nan, arr)
                arr = arr[~np.isnan(arr)]
                if arr.size==0:
                    stats.update({f'b{i}_mean':np.nan, f'b{i}_std':np.nan})
                    continue
                stats[f'b{i}_mean'] = float(arr.mean())
                stats[f'b{i}_std'] = float(arr.std())
                stats[f'b{i}_p10'] = float(np.percentile(arr,10))
                stats[f'b{i}_p90'] = float(np.percentile(arr,90))
    except Exception as e:
        # return empty stats on error to keep pipeline robust
        print('raster_stats error', path, e)
    return stats

def compute_ndvi(path, nir_band_index=8, red_band_index=4):
    # indices should match how bands are ordered in your sentinel composite
    try:
        with rasterio.open(path) as src:
            nir = src.read(nir_band_index).astype('float32')
            red = src.read(red_band_index).astype('float32')
            denom = (nir+red)
            denom[denom==0]=np.nan
            ndvi = (nir-red)/denom
            ndvi = ndvi[~np.isnan(ndvi)]
            return {'ndvi_mean': float(np.mean(ndvi)) if ndvi.size else np.nan, 'ndvi_std': float(np.std(ndvi)) if ndvi.size else np.nan}
    except Exception as e:
        print('compute_ndvi error', path, e)
        return {'ndvi_mean': np.nan, 'ndvi_std': np.nan}

def viirs_stats(path):
    # VIIRS often single-band. Compute mean/std/max.
    try:
        with rasterio.open(path) as src:
            arr = src.read(1).astype('float32')
            arr = arr[arr!=src.nodata] if src.nodata is not None else arr
            arr = arr[~np.isnan(arr)]
            return {'viirs_mean': float(arr.mean()) if arr.size else np.nan, 'viirs_std': float(arr.std()) if arr.size else np.nan, 'viirs_max': float(arr.max()) if arr.size else np.nan}
    except Exception as e:
        print('viirs_stats error', path, e)
        return {'viirs_mean': np.nan, 'viirs_std': np.nan, 'viirs_max': np.nan}

### Example: Extract per-tile stats and merge to tabular
Below is an example pattern; adapt `tile_path` generation for your dataset naming scheme. Extraction is often slow — run with `tqdm` and cache results to CSV.

In [37]:
# Example extraction loop (dry-run / pattern)
# This is a template: adapt 'tile_path' construction to match your files.
def extract_all_stats(df, composite_dir, id_col='tile_id', out_cache='sat_features.csv'):
    # df: table with id_col identifying each tile/row and columns to build filename from (e.g., location, quarter)
    out = []
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        # Prefer explicit filename columns if present (safer). Fall back to constructing a name from geolocation/year/quarter_label.
        if 'sentinel2_tiff_file_name' in row.index and pd.notna(row.get('sentinel2_tiff_file_name')):
            tile_path = composite_dir / row['sentinel2_tiff_file_name']
        else:
            # sanitize geolocation name to match filenames if needed
            loc = row.get('geolocation_name', row.get('data_id', idx))
            q = row.get('quarter_label', row.get('quarter', ''))
            tile_path = composite_dir / f"sentinel_2_{loc}_{q}.tif"
        if tile_path.exists():
            s1 = raster_stats(tile_path)
            s2 = compute_ndvi(tile_path)
            s = {**s1, **s2}
        else:
            s = {}
        s[id_col] = row[id_col] if id_col in row.index else idx
        out.append(s)
    out_df = pd.DataFrame(out)
    out_df.to_csv(out_cache, index=False)
    return out_df

# NOTE: Running this over all tiles can take long; use caching.

## Modeling: GroupKFold + Country-Debias (two-stage)
Approach: compute country mean target, subtract it from `construction_cost_per_m2` to get residuals, train a model to predict residuals using tabular + satellite features, then add country mean back to predictions. Use `GroupKFold` with `location_id` to avoid leakage.

In [38]:
# Prepare features for modeling (example)
# Select numeric features and previously created features. Update as you add satellite features.
target = 'construction_cost_per_m2_usd'
# Determine grouping column: prefer geolocation_name, then data_id, else use index
if 'geolocation_name' in train.columns:
    group_col = 'geolocation_name'
elif 'data_id' in train.columns:
    group_col = 'data_id'
else:
    group_col = None  # fall back to no grouping column

# Exclude identifier / target / time columns. Use 'quarter_label' from dataset.
base_features = [c for c in train.columns if c not in ['construction_cost_per_m2_usd','geolocation_name','data_id','country','year','quarter_label']]
# Filter features to numeric types
base_features = [c for c in base_features if np.issubdtype(train[c].dtype, np.number)]
len(base_features), base_features[:30]

(4,
 ['deflated_gdp_usd',
  'us_cpi',
  'straight_distance_to_capital_km',
  'country_cat'])

In [39]:
# Country de-biasing: subtract country mean from target
train = train.copy()
country_mean = train.groupby('country')[target].transform('mean')
train['target_resid'] = train[target] - country_mean

X = train[base_features].fillna(-999)
y = train['target_resid']
groups = train[group_col].values

# LightGBM CV with GroupKFold
params = {
    'objective':'regression',
    'metric':'mae',
    'learning_rate':0.05,
    'num_leaves':31,
    'verbosity':-1,
    'seed':42
}

gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(X))
scores = []
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
    tr_data = lgb.Dataset(X.iloc[tr_idx], label=y.iloc[tr_idx])
    val_data = lgb.Dataset(X.iloc[val_idx], label=y.iloc[val_idx])
    # Use callbacks for early stopping and to suppress training logs for compatibility across lightgbm versions
    bst = lgb.train(params, tr_data, num_boost_round=2000, valid_sets=[val_data], callbacks=[lgb.early_stopping(100), lgb.log_evaluation(False)])
    pred = bst.predict(X.iloc[val_idx], num_iteration=bst.best_iteration)
    oof[val_idx] = pred
    score = mean_absolute_error(y.iloc[val_idx], pred)
    scores.append(score)
    print(f'Fold {fold} MAE (resid):', score)

print('CV MAE (resid):', np.mean(scores), '±', np.std(scores))

Training until validation scores don't improve for 100 rounds

C:\Users\Prawin\AppData\Local\Temp\ipykernel_39108\2027384240.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  country_mean = train.groupby('country')[target].transform('mean')



Early stopping, best iteration is:
[80]	valid_0's l1: 110.432
Fold 0 MAE (resid): 110.43205868884944
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[102]	valid_0's l1: 169.649
Fold 1 MAE (resid): 169.6487893750427
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[24]	valid_0's l1: 144.91
Fold 2 MAE (resid): 144.9096051852807
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 137.873
Fold 3 MAE (resid): 137.8731844205773
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[47]	valid_0's l1: 92.8043
Fold 4 MAE (resid): 92.8042895908617
CV MAE (resid): 131.13358545212236 ± 26.893530741494928


In [40]:
# Convert residual predictions back to absolute by adding country mean per row and evaluate on train overall (sanity check)
train['pred_resid'] = oof
train['pred_abs'] = train['pred_resid'] + country_mean
mae_abs = mean_absolute_error(train[target], train['pred_abs'])
print('In-sample MAE after adding country mean back:', mae_abs)

In-sample MAE after adding country mean back: 131.17101640511189


## Train final model on full training set and predict on evaluation set
Retrain a model on full training set residuals and produce predictions for the evaluation set. Ensure you compute or load satellite features for evaluation rows too.

In [41]:
# Final training on full data (residual target)
full_data = lgb.Dataset(X, label=y)
final_model = lgb.train(params, full_data, num_boost_round=bst.best_iteration if 'bst' in globals() else 1000)

# Prepare eval features (must match base_features).
# Build X_eval by ensuring the same columns as base_features. If a feature is missing in eval_df, fill with -999.
eval_cols = []
missing = []
for f in base_features:
    if f in eval_df.columns:
        eval_cols.append(f)
    else:
        missing.append(f)
if missing:
    print('Warning: the following features are missing in eval_df and will be filled with -999:', missing)
# Create X_eval with same column order as base_features
X_eval = pd.DataFrame(index=eval_df.index)
for f in base_features:
    if f in eval_df.columns:
        X_eval[f] = eval_df[f].fillna(-999)
    else:
        X_eval[f] = -999
pred_eval_resid = final_model.predict(X_eval)
# Add country mean from train (map by country). For countries not in train, use global mean.
country_map = train.groupby('country')[target].mean().to_dict()
global_mean = train[target].mean()
# ensure we don't try to insert numeric values into a categorical column (convert to object first)
eval_country_mean = eval_df['country'].astype(object).map(country_map).fillna(global_mean)
pred_eval_abs = pred_eval_resid + eval_country_mean.values

# Prepare submission (adapt column names to competition requirement)
submission = eval_df.copy()
submission['construction_cost_per_m2_usd'] = pred_eval_abs
submission[['geolocation_name','year','quarter_label','construction_cost_per_m2_usd']].head()

# Save submission
submission[['geolocation_name','year','quarter_label','construction_cost_per_m2_usd']].to_csv(BASE / 'submission.csv', index=False)
print('Saved submission to', BASE / 'submission.csv')

Saved submission to d:\Projects\Construction-cost\submission.csv


C:\Users\Prawin\AppData\Local\Temp\ipykernel_39108\133379843.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  country_map = train.groupby('country')[target].mean().to_dict()


## Enhanced Pipeline: Satellite caching, features, CatBoost & stacking
This section adds: satellite processing + caching, derived multispectral features (NDVI, NDBI, urban index, texture), VIIRS percentiles, a CatBoost baseline trained on residuals, and a simple stacking ensemble (Ridge) combining LightGBM + CatBoost. It also saves OOF predictions and feature importances for analysis. Run cells in order.

In [42]:
# Enhanced satellite processing + caching
from pathlib import Path
import hashlib

def compute_tile_features(s_path, v_path):
    # s_path: sentinel tiff path (can be None), v_path: viirs tiff path (can be None)
    feats = {}
    if s_path is not None and Path(s_path).exists():
        s1 = raster_stats(s_path)
        s2 = compute_ndvi(s_path)
        # compute a simple built-up index (NDBI) and urban index if bands exist
        try:
            with rasterio.open(s_path) as src:
                idxs = src.indexes
                # common Sentinel naming: red ~4, nir ~8, swir ~11 (indexes may vary)
                b4 = src.read(4).astype('float32') if 4 in idxs else None
                b8 = src.read(8).astype('float32') if 8 in idxs else None
                b11 = src.read(11).astype('float32') if 11 in idxs else None
                if b8 is not None and b4 is not None:
                    denom = (b8 + b4)
                    denom[denom==0] = np.nan
                    urban = (b8 - b4) / denom
                    feats['urban_mean'] = float(np.nanmean(urban))
                    feats['urban_std'] = float(np.nanstd(urban))
                if b11 is not None and b8 is not None:
                    denom2 = (b11 + b8)
                    denom2[denom2==0] = np.nan
                    ndbi = (b11 - b8) / denom2
                    feats['ndbi_mean'] = float(np.nanmean(ndbi))
        except Exception as e:
            pass
        feats.update(s1)
        feats.update(s2)
    if v_path is not None and Path(v_path).exists():
        try:
            with rasterio.open(v_path) as src:
                arr = src.read(1).astype('float32')
                arr = arr[~np.isnan(arr)]
                feats['viirs_p50'] = float(np.nanpercentile(arr,50)) if arr.size else np.nan
                feats['viirs_p90'] = float(np.nanpercentile(arr,90)) if arr.size else np.nan
                feats['viirs_mean'] = float(arr.mean()) if arr.size else np.nan
        except Exception as e:
            pass
    return feats

def build_sat_cache(df, composite_dir, cache_path):
    cache_path = Path(cache_path)
    if cache_path.exists():
        return pd.read_csv(cache_path)
    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df)):
        sfile = r.get('sentinel2_tiff_file_name') if 'sentinel2_tiff_file_name' in r.index else None
        vfile = r.get('viirs_tiff_file_name') if 'viirs_tiff_file_name' in r.index else None
        s_path = (composite_dir / sfile) if sfile is not None else None
        v_path = (composite_dir / vfile) if vfile is not None else None
        feats = compute_tile_features(s_path, v_path)
        feats['index'] = r.name
        rows.append(feats)
    out = pd.DataFrame(rows)
    out.to_csv(cache_path, index=False)
    return out

# Example usage (cache files into datasets/):
# sat_train = build_sat_cache(train, SENTINEL_DIR, DATA_DIR / 'sat_features_train.csv')
# sat_eval = build_sat_cache(eval_df, SENTINEL_DIR, DATA_DIR / 'sat_features_eval.csv')

In [43]:
# Merge cached satellite features if available (example)
sat_train_path = DATA_DIR / 'sat_features_train.csv'
sat_eval_path = DATA_DIR / 'sat_features_eval.csv'
if sat_train_path.exists():
    sat_train = pd.read_csv(sat_train_path)
    sat_train = sat_train.set_index('index')
    train = pd.concat([train.reset_index(drop=True), sat_train.reindex(train.index).reset_index(drop=True)], axis=1)
if sat_eval_path.exists():
    sat_eval = pd.read_csv(sat_eval_path)
    sat_eval = sat_eval.set_index('index')
    eval_df = pd.concat([eval_df.reset_index(drop=True), sat_eval.reindex(eval_df.index).reset_index(drop=True)], axis=1)

print('After merge train columns:', train.shape, 'eval columns:', eval_df.shape)

After merge train columns: (1024, 28) eval columns: (1024, 24)


In [44]:
# Map categorical risk levels to numeric (example)
risk_map = {'Very Low':0, 'Low':1, 'Moderate':2, 'High':3, 'Very High':4}
if 'tropical_cyclone_wind_risk' in train.columns:
    train['tropical_cyclone_wind_risk_num'] = train['tropical_cyclone_wind_risk'].map(risk_map).fillna(-1)
    eval_df['tropical_cyclone_wind_risk_num'] = eval_df['tropical_cyclone_wind_risk'].map(risk_map).fillna(-1)

# Recompute base_features after merging satellite columns
base_features = [c for c in train.columns if c not in ['construction_cost_per_m2_usd','geolocation_name','data_id','country','year','quarter_label','target_resid','pred_resid','pred_abs']]
base_features = [c for c in base_features if np.issubdtype(train[c].dtype, np.number)]
len(base_features)

5

In [45]:
# Train CatBoost on residuals with GroupKFold (if catboost is installed)
try:
    from catboost import CatBoostRegressor
    cat_installed = True
except Exception:
    cat_installed = False

if cat_installed:
    cat_oof = np.zeros(len(train))
    cat_models = []
    cat_params = dict(iterations=1000, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
    cat_features = [f for f in ['country_cat','climate_zone_cat'] if f in train.columns]
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(train, train['target_resid'], groups=train[group_col] if group_col in train.columns else None)):
        trX = train.iloc[tr_idx][base_features]
        valX = train.iloc[val_idx][base_features]
        trY = train.iloc[tr_idx]['target_resid']
        valY = train.iloc[val_idx]['target_resid']
        model = CatBoostRegressor(**cat_params)
        model.fit(trX, trY, eval_set=(valX, valY), cat_features=[i for i,f in enumerate(base_features) if f in cat_features] if cat_features else None, early_stopping_rounds=50)
        pred = model.predict(valX)
        cat_oof[val_idx] = pred
        cat_models.append(model)
        print('CatBoost fold', fold, 'MAE:', mean_absolute_error(valY, pred))
    print('CatBoost CV MAE (resid):', mean_absolute_error(train['target_resid'], cat_oof))
else:
    print('CatBoost not installed; skip CatBoost training. Install with `pip install catboost`.')

CatBoost not installed; skip CatBoost training. Install with `pip install catboost`.


In [46]:
# Simple stacking of residual predictors (LGBM oof `oof` already exists from earlier)
from sklearn.linear_model import Ridge
stack_X = pd.DataFrame({'lgb_resid': oof})
if 'cat_oof' in globals():
    stack_X['cat_resid'] = cat_oof
# train meta-learner on residuals
meta = Ridge(alpha=1.0)
meta.fit(stack_X, train['target_resid'])
stack_pred = meta.predict(stack_X)
print('Stack MAE on residuals (train):', mean_absolute_error(train['target_resid'], stack_pred))
# Convert stacked residuals to absolute by adding country mean back
train['pred_stack_resid'] = stack_pred
train['pred_stack_abs'] = train['pred_stack_resid'] + country_mean
print('Stack absolute MAE (train):', mean_absolute_error(train[target], train['pred_stack_abs']))

Stack MAE on residuals (train): 134.00022518244782
Stack absolute MAE (train): 134.0002251824478


In [47]:
# Feature importances: LGBM and CatBoost (if available)
importances = {}
if 'bst' in globals():
    imp = bst.feature_importance(importance_type='gain')
    imps = pd.DataFrame({'feature': X.columns, 'importance': imp}).sort_values('importance', ascending=False)
    (BASE / 'artifacts').mkdir(exist_ok=True)
    imps.to_csv(BASE / 'artifacts' / 'lgbm_feature_importance.csv', index=False)
    importances['lgbm'] = imps
if 'cat_models' in globals() and len(cat_models)>0:
    cimp = cat_models[0].get_feature_importance()
    cimps = pd.DataFrame({'feature': base_features, 'importance': cimp}).sort_values('importance', ascending=False)
    cimps.to_csv(BASE / 'artifacts' / 'catboost_feature_importance.csv', index=False)
    importances['catboost'] = cimps
print('Saved feature importances to artifacts/*.')

Saved feature importances to artifacts/*.


In [48]:
# Save OOF predictions for analysis
(BASE / 'artifacts').mkdir(exist_ok=True)
oof_df = train[[ 'geolocation_name','year','quarter_label']].copy()
oof_df['target'] = train[target]
oof_df['lgb_resid_oof'] = oof
if 'cat_oof' in globals():
    oof_df['cat_resid_oof'] = cat_oof
if 'train' in globals() and 'pred_stack_abs' in train.columns:
    oof_df['stack_pred_abs'] = train['pred_stack_abs']
oof_df.to_csv(BASE / 'artifacts' / 'oof_predictions.csv', index=False)
print('Saved OOF to artifacts/oof_predictions.csv')

Saved OOF to artifacts/oof_predictions.csv


In [49]:
# Ensemble/stacking: combine LGBM + CatBoost (optional XGBoost placeholder)
from sklearn.linear_model import Ridge
# Build stack training frame from OOFs available: `oof` for LGBM, `cat_oof` for CatBoost (if trained)
stack_cols = {}
stack_cols['lgb_resid_oof'] = oof
if 'cat_oof' in globals():
    stack_cols['cat_resid_oof'] = cat_oof
# XGBoost placeholder: if you add xgb_oof, include here
stack_train = pd.DataFrame(stack_cols)
# Meta target: residuals
meta_y = train['target_resid'].values
# Fit a simple Ridge meta-learner with cross-validation (no leakage since OOFs used)
meta = Ridge(alpha=1.0)
meta.fit(stack_train, meta_y)
meta_pred = meta.predict(stack_train)
print('Meta MAE on residuals:', mean_absolute_error(meta_y, meta_pred))
# Create eval stack features: predict_eval_resid from LGBM + CatBoost ensemble
eval_stack = pd.DataFrame()
# LGBM eval preds already computed as `pred_eval_resid` earlier in file; if not, compute now using `final_model`
if 'pred_eval_resid' not in globals():
    pred_eval_resid = final_model.predict(X_eval)
eval_stack['lgb_resid_oof'] = pred_eval_resid
# CatBoost ensemble on eval: average predictions from cat_models if present
if 'cat_models' in globals() and len(cat_models)>0:
    cat_preds = np.column_stack([m.predict(X_eval[base_features]) for m in cat_models])
    eval_stack['cat_resid_oof'] = cat_preds.mean(axis=1)
# Meta predict on eval
pred_eval_stack_resid = meta.predict(eval_stack)
# Add country mean back to stacked residuals
if 'country_map' not in globals():
    country_map = train.groupby('country')[target].mean().to_dict()
global_mean = train[target].mean()
eval_country_mean = eval_df['country'].astype(object).map(country_map).fillna(global_mean)
pred_eval_stack_abs = pred_eval_stack_resid + eval_country_mean.values
# Save ensembled submission
ens_submission = eval_df[[ 'geolocation_name','year','quarter_label']].copy()
ens_submission['construction_cost_per_m2_usd'] = pred_eval_stack_abs
ens_submission.to_csv(BASE / 'submission_ensemble.csv', index=False)
print('Saved ensemble submission to', BASE / 'submission_ensemble.csv')

Meta MAE on residuals: 134.00022518244782
Saved ensemble submission to d:\Projects\Construction-cost\submission_ensemble.csv


In [50]:
# Hyperparameter tuning (optional, uses Optuna). Controlled by RUN_TUNING flag to avoid long runs.
RUN_TUNING = False  # set to True to run a short tuning job
TUNING_TRIALS = 20

if RUN_TUNING:
    try:
        import optuna
    except Exception:
        print('Optuna not installed. Install with `pip install optuna` to enable tuning. Skipping tuning.')
        RUN_TUNING = False

if RUN_TUNING:
    def objective_lgb(trial):
        param = {
            'objective':'regression',
            'metric':'mae',
            'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.2),
            'num_leaves': trial.suggest_int('num_leaves', 16, 128),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 200),
            'feature_fraction': trial.suggest_uniform('feature_fraction', 0.4, 1.0),
            'bagging_fraction': trial.suggest_uniform('bagging_fraction', 0.4, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 0, 10),
            'verbosity': -1,
            'seed': 42
        }
        # simple CV using GroupKFold and small num_boost_round to speed tuning
        cv_scores = []
        for tr_idx, val_idx in gkf.split(X, y, groups=train[group_col] if group_col in train.columns else None):
            tr_data = lgb.Dataset(X.iloc[tr_idx], label=y.iloc[tr_idx])
            val_data = lgb.Dataset(X.iloc[val_idx], label=y.iloc[val_idx])
            bst_local = lgb.train(param, tr_data, num_boost_round=500, valid_sets=[val_data], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(False)])
            pred = bst_local.predict(X.iloc[val_idx], num_iteration=bst_local.best_iteration)
            cv_scores.append(mean_absolute_error(y.iloc[val_idx], pred))
        return np.mean(cv_scores)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective_lgb, n_trials=TUNING_TRIALS)
    print('Best LGB params:', study.best_params, 'best value (MAE):', study.best_value)
    best_lgb_params = study.best_params
    # merge with defaults
    params.update(best_lgb_params)

    # Optional CatBoost tuning if installed
    if cat_installed:
        def objective_cat(trial):
            param = {
                'iterations': trial.suggest_int('iterations', 200, 2000),
                'learning_rate': trial.suggest_loguniform('cat_lr', 1e-3, 0.2),
                'depth': trial.suggest_int('depth', 4, 10),
                'random_seed': 42,
                'verbose': 0
            }
            cv_scores = []
            for tr_idx, val_idx in gkf.split(train, train['target_resid'], groups=train[group_col] if group_col in train.columns else None):
                trX = train.iloc[tr_idx][base_features]
                valX = train.iloc[val_idx][base_features]
                trY = train.iloc[tr_idx]['target_resid']
                valY = train.iloc[val_idx]['target_resid']
                model = CatBoostRegressor(**param)
                model.fit(trX, trY, eval_set=(valX, valY), early_stopping_rounds=50)
                pred = model.predict(valX)
                cv_scores.append(mean_absolute_error(valY, pred))
            return np.mean(cv_scores)
        study_cat = optuna.create_study(direction='minimize')
        study_cat.optimize(objective_cat, n_trials=max(10, TUNING_TRIALS//2))
        print('Best Cat params:', study_cat.best_params, 'best value (MAE):', study_cat.best_value)
        best_cat_params = study_cat.best_params
else:
    print('Tuning skipped. Set RUN_TUNING = True and install optuna to run tuning.')

Tuning skipped. Set RUN_TUNING = True and install optuna to run tuning.


In [51]:
# Quick runner cell: toggle tuning or ensemble-only run
# Set to True to run hyperparameter tuning (may take long).
RUN_TUNING = True
if RUN_TUNING:
    print('Running tuning...')
else:
    print('Skipping tuning. Ensemble already computed in prior cell and saved to submission_ensemble.csv')

Running tuning...


## Next steps & competition tips
- Carefully compute and cache satellite statistics for train and evaluation sets (this is the most useful signal).
- Create features: NDVI, NDBI, urban indices, VIIRS percentiles, texture metrics.
- Use CatBoost for native categoricals and compare with LightGBM.
- Build an ensemble (LGBM + CatBoost + XGBoost) and use out-of-fold stacking.
- Perform error analysis by country and location; look for systematic biases.
- Be strict with validation: GroupKFold by `location_id`. Do not random-split.

If you'd like, I can: provide a full satellite-processing script, implement caching, or add an ensemble + hyperparameter tuning cell. Which would you like next?